# Coverage-Guided Fuzz Test Input Generation (SuT: `calculate_shipping_fee`)

In this exercise, we'll implement a simple **coverage-guided fuzzing** algorithm that generates random test inputs and selects those that increase structural coverages.

In [1]:
import os, sys

# exercises → unit folder → modules → project_root
MODULES_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CODE_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "code")
MODEL_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "model")
DATA_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "data")

if CODE_DIR not in sys.path:
    sys.path.append(CODE_DIR)

if MODEL_DIR not in sys.path:
    sys.path.append(MODEL_DIR)

if DATA_DIR not in sys.path:
    sys.path.append(DATA_DIR)

print("Added to sys.path:", CODE_DIR)
print("Added to sys.path:", MODEL_DIR)
print("Added to sys.path:", DATA_DIR)

from shipping_fee_instrumentation import *

stmt_tracker = StatementCoverageTracker()
branch_tracker = BranchCoverageTracker()
path_tracker = PathCoverageTracker()

print("Three structural coverage trackers have been initialized.")

Added to sys.path: /workspace/modules/exercise_artifacts/code
Added to sys.path: /workspace/modules/exercise_artifacts/model
Added to sys.path: /workspace/modules/exercise_artifacts/data
Three structural coverage trackers have been initialized.


## Random Input Generator

In [2]:
# Pure random generator (baseline)
import random

def generate_random_test_input():
    order_total = random.randint(0, 120000)
    weight_kg = round(random.uniform(0, 50), 2)
    distance_km = random.randint(0, 200)
    is_island = random.choice([True, False])
    membership = random.choice(["NONE", "WOW"])
    coupon_type = random.choice(["NONE", "NEW_USER"])
    return order_total, weight_kg, distance_km, is_island, membership, coupon_type

## Statement Coverage-Guided Generation

In [3]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    stmt_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        stmt_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    stmt_tracker.run_test(*random_test_input)
    new_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
stmt_tracker.reset()
for test_input in test_inputs:
    stmt_tracker.run_test(*test_input)
stmt_tracker.print_report()

STATEMENT COVERAGE REPORT

Overall Coverage: 100.00% (12/12 items)
Total Tests: 5

Test Cases:
1: (12969, 39.3, 17, False, 'WOW', 'NONE')
2: (96525, 24.94, 159, True, 'WOW', 'NEW_USER')
3: (45536, 8.9, 94, True, 'WOW', 'NONE')
4: (119864, 2.39, 24, True, 'WOW', 'NEW_USER')
5: (89726, 6.27, 5, False, 'WOW', 'NEW_USER')

| Test input             | 1    | 2    | 3    | 4   | 5   | Covered   |
|------------------------|------|------|------|-----|-----|-----------|
| fee = 0                | O    | O    | O    | O   | O   | O         |
| fee += 3000            | O    | X    | X    | X   | X   | O         |
| fee += 0 (weight)      | X    | X    | X    | O   | X   | O         |
| fee += 2000            | X    | X    | O    | X   | O   | O         |
| fee += 5000            | O    | O    | X    | X   | X   | O         |
| fee += 0 (distance)    | X    | X    | X    | X   | O   | O         |
| fee += 1000            | O    | X    | X    | O   | X   | O         |
| fee += 3000 (distance) | X   

## Branch Coverage-Guided Generation

In [4]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    branch_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        branch_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    branch_tracker.run_test(*random_test_input)
    new_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
branch_tracker.reset()
for test_input in test_inputs:
    branch_tracker.run_test(*test_input)
branch_tracker.print_report()

BRANCH COVERAGE REPORT

Overall Coverage: 100.00% (16/16 items)
Total Tests: 5

Test Cases:
1: (14431, 1.81, 43, False, 'NONE', 'NEW_USER')
2: (10601, 48.97, 10, True, 'NONE', 'NONE')
3: (17156, 40.94, 184, False, 'NONE', 'NEW_USER')
4: (94587, 17.02, 185, False, 'NONE', 'NEW_USER')
5: (39991, 3.91, 92, False, 'WOW', 'NEW_USER')

| Test input                     | 1    | 2     | 3    | 4    | 5    | Covered   |
|--------------------------------|------|-------|------|------|------|-----------|
| order_total < 40000: True      | O    | O     | O    | X    | O    | O         |
| order_total < 40000: False     | X    | X     | X    | O    | X    | O         |
| weight_kg <= 5: True           | O    | X     | X    | X    | O    | O         |
| weight_kg <= 5: False          | X    | O     | O    | O    | X    | O         |
| weight_kg <= 20: True          | X    | X     | X    | O    | X    | O         |
| weight_kg <= 20: False         | X    | O     | O    | X    | X    | O         |
| di

## Path Coverage-Guided Generation

In [5]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    path_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        path_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    path_tracker.run_test(*random_test_input)
    new_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
path_tracker.reset()
for test_input in test_inputs:
    path_tracker.run_test(*test_input)
path_tracker.print_report()

PATH COVERAGE REPORT

Overall Coverage: 37.50% (54/144 items)
Total Tests: 54

Test Cases:
1: (102893, 47.86, 100, False, 'NONE', 'NONE')
2: (54090, 18.53, 44, True, 'WOW', 'NONE')
3: (91167, 44.47, 146, True, 'WOW', 'NEW_USER')
4: (18158, 45.14, 26, False, 'NONE', 'NEW_USER')
5: (98282, 28.17, 102, False, 'NONE', 'NEW_USER')
6: (110684, 12.69, 134, False, 'NONE', 'NONE')
7: (46877, 49.65, 101, False, 'WOW', 'NEW_USER')
8: (67559, 4.83, 122, False, 'WOW', 'NEW_USER')
9: (91580, 4.42, 10, True, 'WOW', 'NONE')
10: (100565, 44.14, 108, True, 'WOW', 'NONE')
11: (48295, 46.55, 96, True, 'NONE', 'NEW_USER')
12: (26671, 13.27, 46, True, 'WOW', 'NONE')
13: (66639, 25.0, 1, True, 'WOW', 'NEW_USER')
14: (60959, 10.4, 180, True, 'NONE', 'NEW_USER')
15: (48583, 3.12, 70, True, 'NONE', 'NONE')
16: (99756, 35.23, 70, True, 'NONE', 'NONE')
17: (73339, 17.24, 22, False, 'WOW', 'NONE')
18: (1703, 2.94, 125, True, 'WOW', 'NEW_USER')
19: (34589, 24.43, 20, True, 'WOW', 'NONE')
20: (114525, 44.0, 83, Fals

## 🤔 Think: Smarter Random Input Generator

Our `pure` random input generator struggles to achieve high coverage.

**Why?** Uniform random inputs across six parameters often miss edge combinations that trigger meaningful branches (e.g., threshold boundaries, island surcharge, membership/coupon interactions).

---

### ✅ Your Task

Modify the `generate_random_test_input()` above to include **stronger bias toward boundary values** for:
- `order_total` around `40000`
- `weight_kg` around `5` and `20`
- `distance_km` around `10` and `50`
- categorical combinations of `membership` / `coupon_type`

This simple biasing strategy can significantly improve structural coverage.

---

## 🎯 Smarter Random Generators in Fuzzing

The ultimate goal of fuzzing is not just to generate random inputs —  
but to generate **smarter random inputs**.

Below are representative approaches used in practice:

| Approach                         | Required SuT Knowledge                     | Example                          |
|----------------------------------|--------------------------------------------|----------------------------------|
| Pure Random                      | 🟢 None (Black-box)                         | Uniform random sampling          |
| Biased Random                    | 🟢 Very Low (general heuristics only)       | Small-int bias, boundary values  |
| Mutation-Based                   | 🟡 Low (seed inputs required)               | AFL-style bit/byte mutations     |
| Evolutionary / Genetic           | 🟡 Grey-box (fitness such as coverage)      | Coverage-based genetic fuzzing   |
| Grammar-Based                    | 🟠 Input format / grammar knowledge         | JSON/XML grammar fuzzing         |
| Constraint-Based / Symbolic      | 🔴 High (White-box, path constraint access) | KLEE, symbolic execution engines |


---

### 🔎 Reflection

- Why do boundary-focused inputs improve coverage in this case?
- What kind of bias would help if the input domain were images? Strings? Network packets?
- How much knowledge about the system under test (SuT) should a fuzzing strategy assume?

Keep in mind:

> Pure random is rarely efficient in practice.  
> **The key to effective fuzzing is choosing (or designing) a strategy that fits your SuT.**